# Typed Failure Recovery for Neuro-Symbolic Reasoning

## What This Notebook Does

This demo shows a **typed failure repair pipeline** for logical reasoning over natural language. The system:

1. **Parses natural language** (ProofWriter theories and CLUTRR kinship stories) into first-order logic (FOL)
2. **Executes queries** using a Python forward-chaining logic engine
3. **Classifies failures** into three types:
   - **Type 1**: Unknown predicate (vocabulary gap)
   - **Type 2**: Arity mismatch (wrong argument count)
   - **Type 3**: Missing fact (incomplete knowledge base)
4. **Applies targeted repairs** via LLM prompts specific to each failure type
5. **Compares** against a generic baseline repair prompt

## Key Finding

The typed pipeline achieves **zero hallucination** (0.0%) while the baseline produces hallucinations in 8% of cases. Both achieve similar accuracy (24% typed vs 26% baseline), suggesting the main barrier is natural language-to-FOL extraction rather than repair strategy.

## Run This Notebook

This is a **demo with 6 examples** (3 ProofWriter + 3 CLUTRR). The original experiment ran 50 examples; you can scale up by modifying the config cell below.


In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Non-Colab packages (always install)
_pip('loguru==0.7.3', 'tenacity==9.1.4')

# Core packages (pre-installed on Colab, install locally only)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')


In [ ]:
import json
import os
import re
from pathlib import Path
from typing import Optional, Any
import sys

from loguru import logger
from tenacity import retry, stop_after_attempt, wait_exponential

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Suppress verbose logging for the demo
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{message}")


In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-7fd9e8-typed-unification-failure-recovery-a-str/main/round-2/experiment-1/demo/mini_demo_data.json"
import urllib.request

def load_data():
    '''Load demo data from GitHub URL with local fallback.'''
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        logger.warning(f"GitHub URL failed ({e}), trying local file...")
    
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local path")


In [ ]:
data = load_data()
logger.info(f"Loaded demo data: {data['metadata']['total_examples']} examples across {len(data['datasets'])} datasets")
print(f"\nMetadata:\n{json.dumps(data['metadata'], indent=2)}")


## Configuration

These parameters control the demo scale. Start at minimum values; increase gradually if needed.


In [ ]:
# Demo configuration - ABSOLUTE MINIMUM
MAX_EXAMPLES_TO_SHOW = 3  # Show first 3 examples from data
VISUALIZE = True  # Show plots
VERBOSE = True  # Print detailed trace info

logger.info(f"Demo config: show {MAX_EXAMPLES_TO_SHOW} examples, visualize={VISUALIZE}, verbose={VERBOSE}")


## 1. Forward-Chaining Logic Engine

The engine evaluates FOL facts and rules using bottom-up forward chaining.

- **Facts**: Ground atoms like `needs(dog, bear)`
- **Rules**: Horn clauses like `prop(?X, rough) :- needs(?X, bear)`
- **Query**: Prove a goal by saturating all derivable facts


In [ ]:
class Predicate:
    '''Ground atom: name + tuple of string arguments.'''
    __slots__ = ("name", "args")

    def __init__(self, name: str, args: tuple):
        self.name = name
        self.args = args

    def __eq__(self, other):
        return isinstance(other, Predicate) and self.name == other.name and self.args == other.args

    def __hash__(self):
        return hash((self.name, self.args))

    def __repr__(self):
        return f"{self.name}({', '.join(self.args)})"


class Rule:
    '''Horn clause: head :- body (list of Predicates with possible variables).'''
    __slots__ = ("head", "body")

    def __init__(self, head, body):
        self.head = head
        self.body = body


class LogicEngine:
    '''Simple bottom-up forward-chaining with unification for Horn clauses.'''

    def __init__(self):
        self.facts = set()
        self.rules = []
        self.predicate_arities = {}

    def add_fact(self, name: str, args: tuple) -> None:
        p = Predicate(name, args)
        self.facts.add(p)
        self.predicate_arities.setdefault(name, set()).add(len(args))

    def add_rule(self, head, body) -> None:
        self.rules.append(Rule(head, body))
        name, args = head
        self.predicate_arities.setdefault(name, set()).add(len(args))

    def parse_and_add(self, clause: str) -> None:
        '''Parse 'pred(a, b).' or 'head(X) :- body(X, Y).' and add.'''
        clause = clause.strip().rstrip(".")
        if ":-" in clause:
            head_str, body_str = clause.split(":-", 1)
            head = _parse_term(head_str.strip())
            body = [_parse_term(t.strip()) for t in _split_body(body_str.strip())]
            self.add_rule(head, body)
        else:
            name, args = _parse_term(clause)
            self.add_fact(name, args)

    def _unify(self, pattern, ground, bindings):
        '''Unify pattern with ground fact. Returns updated bindings or None.'''
        name_p, args_p = pattern
        name_g, args_g = ground
        if name_p != name_g or len(args_p) != len(args_g):
            return None
        new_b = dict(bindings)
        for pv, gv in zip(args_p, args_g):
            if pv.startswith("?"):
                if pv in new_b:
                    if new_b[pv] != gv:
                        return None
                else:
                    new_b[pv] = gv
            elif pv != gv:
                return None
        return new_b

    def _apply_bindings(self, term, bindings):
        name, args = term
        resolved = tuple(bindings.get(a, a) for a in args)
        return name, resolved

    def forward_chain(self, max_iters=20) -> None:
        '''Saturate facts by applying all rules repeatedly.'''
        for _ in range(max_iters):
            new_facts = set()
            for rule in self.rules:
                self._apply_rule(rule, new_facts)
            added = new_facts - self.facts
            if not added:
                break
            self.facts |= added

    def _apply_rule(self, rule, new_facts) -> None:
        '''Try to fire rule; add derived facts.'''
        self._match_body(rule.body, {}, rule, new_facts)

    def _match_body(self, body, bindings, rule, out) -> None:
        if not body:
            name, args = self._apply_bindings(rule.head, bindings)
            if all(not a.startswith("?") for a in args):
                out.add(Predicate(name, args))
            return
        lit = body[0]
        rest = body[1:]
        for fact in self.facts:
            b2 = self._unify(lit, (fact.name, fact.args), bindings)
            if b2 is not None:
                self._match_body(rest, b2, rule, out)

    def query(self, name: str, args: tuple):
        '''Returns (success, bindings_list, error_message).'''
        # Check for unknown predicate
        if name not in self.predicate_arities:
            return False, [], f"existence_error(procedure,{name}/{len(args)})"
        # Check for arity mismatch
        expected_arities = self.predicate_arities[name]
        if len(args) not in expected_arities:
            expected = sorted(expected_arities)[0]
            return False, [], f"arity_error({name},{expected},{len(args)})"
        # Run forward chaining
        self.forward_chain()
        results = []
        for fact in self.facts:
            if fact.name != name or len(fact.args) != len(args):
                continue
            match = {}
            ok = True
            for pat, val in zip(args, fact.args):
                if pat is None:
                    match[val] = val
                elif pat != val:
                    ok = False
                    break
            if ok:
                results.append(match)
        if results:
            return True, results, ""
        return False, [], "missing_fact"


def _parse_term(s: str):
    '''Parse 'pred(arg1, arg2)' → ('pred', ('arg1', 'arg2')).'''
    s = s.strip()
    m = re.match(r"^(\w+)\(([^)]*)\)$", s)
    if m:
        name = m.group(1)
        raw_args = m.group(2)
        args = tuple(a.strip() for a in raw_args.split(",") if a.strip())
        return name, args
    m2 = re.match(r"^(\w+)$", s)
    if m2:
        return m2.group(1), ()
    return s, ()


def _split_body(body_str: str):
    '''Split comma-separated body literals (respecting parentheses).'''
    parts, depth, cur = [], 0, ""
    for ch in body_str:
        if ch == "(":
            depth += 1
            cur += ch
        elif ch == ")":
            depth -= 1
            cur += ch
        elif ch == "," and depth == 0:
            parts.append(cur.strip())
            cur = ""
        else:
            cur += ch
    if cur.strip():
        parts.append(cur.strip())
    return parts

logger.info("Logic engine classes defined")


## 2. Natural Language to FOL Extraction

Convert ProofWriter theories and CLUTRR kinship stories into FOL predicates.

**ProofWriter**: Transforms sentences like "The dog needs the bear" → `needs(dog, bear)`

**CLUTRR**: Extracts kinship relations like "[Alice]'s father [Bob]" → `father(alice, bob)` + inference rules


In [ ]:
def _normalize(s: str) -> str:
    '''Normalize entity name to valid Prolog atom.'''
    return re.sub(r"\s+", "_", s.strip().lower().replace("-", "_"))


def extract_fol_simple(text: str, dataset_name: str):
    '''Extract FOL facts from text. Simplified for demo.'''
    if "Theory:" in text or dataset_name == "proofwriter":
        # ProofWriter: extract simple facts from theory
        theory_m = re.search(r"Theory:\s*(.*?)(?:\nQuery:|$)", text, re.DOTALL)
        theory_text = theory_m.group(1).strip() if theory_m else text
        
        facts = []
        sentences = re.split(r"\.\s+", theory_text)
        for sent in sentences[:5]:  # Limit to 5 sentences for demo
            sent = sent.strip().rstrip(".")
            if not sent or "not" in sent.lower():
                continue
            # "The X verbs the Y."
            m = re.match(r"^The ([\w ]+?) (needs?|chases?) the ([\w ]+)$", sent, re.IGNORECASE)
            if m:
                subj, rel, obj = _normalize(m.group(1)), _normalize(m.group(2).rstrip("s")), _normalize(m.group(3))
                facts.append(f"{rel}({subj},{obj})")
            # "The X is Y."
            m = re.match(r"^The ([\w ]+?) is (\w+)$", sent, re.IGNORECASE)
            if m:
                subj, prop = _normalize(m.group(1)), _normalize(m.group(2))
                facts.append(f"prop({subj},{prop})")
        return facts, "query", ()
    
    elif "Story:" in text or dataset_name == "clutrr":
        # CLUTRR: extract kinship facts
        story_m = re.search(r"Story:\s*(.*?)(?:\nQuery:|$)", text, re.DOTALL)
        story_text = story_m.group(1).strip() if story_m else text
        
        facts = []
        for m in re.finditer(r"\[(\w+)\]'s (father|mother|son|daughter|brother|sister),?\s+\[(\w+)\]", story_text, re.IGNORECASE):
            a, rel, b = m.group(1).lower(), m.group(2).lower(), m.group(3).lower()
            facts.append(f"{rel}({a},{b})")
        
        # Add basic kinship rules
        facts.extend([
            "parent(?X,?Y) :- father(?X,?Y)",
            "parent(?X,?Y) :- mother(?X,?Y)",
            "grandfather(?X,?Z) :- father(?X,?Y),parent(?Y,?Z)",
        ])
        return facts, "any_kinship", ()
    
    return [], "query", ()

logger.info("FOL extraction functions defined")


## 3. Failure Classification

When a query fails, classify the error type:

- **Type 1** (Unknown Predicate): The predicate doesn't exist in the knowledge base
- **Type 2** (Arity Mismatch): Wrong number of arguments
- **Type 3** (Missing Fact): Predicate exists but fact can't be proven


In [ ]:
def classify_failure(error_msg: str) -> str:
    '''Classify failure into Type 1/2/3/other.'''
    if "existence_error" in error_msg or "unknown_pred" in error_msg:
        return "type1_unknown_predicate"
    if "arity_error" in error_msg:
        return "type2_arity_mismatch"
    if error_msg == "missing_fact" or not error_msg:
        return "type3_missing_fact"
    return "other"

logger.info("Failure classification defined")


## 4. Demo Processing

Load the demo data and show results from the original 50-example run.


In [ ]:
# Process demo data
results_by_dataset = {}
for dataset_info in data['datasets']:
    ds_name = dataset_info['dataset']
    examples = dataset_info['examples'][:MAX_EXAMPLES_TO_SHOW]
    
    results_by_dataset[ds_name] = {
        'typed_correct': sum(1 for ex in examples if ex.get('metadata_typed_correct') == 'True'),
        'baseline_correct': sum(1 for ex in examples if ex.get('metadata_baseline_correct') == 'True'),
        'typed_repairs': sum(1 for ex in examples if ex.get('metadata_typed_repair_applied') == 'True'),
        'baseline_repairs': sum(1 for ex in examples if ex.get('metadata_baseline_repair_applied') == 'True'),
        'examples': examples,
    }

logger.info(f"Loaded {sum(len(v['examples']) for v in results_by_dataset.values())} demo examples")

# Print sample
for ds_name, results in results_by_dataset.items():
    logger.info(f"\n{ds_name.upper()}: {len(results['examples'])} examples")
    logger.info(f"  Typed correct: {results['typed_correct']}/{len(results['examples'])}")
    logger.info(f"  Baseline correct: {results['baseline_correct']}/{len(results['examples'])}")


## 5. Full Experiment Results

Results from the complete 50-example run (this notebook shows 6 demo examples).


In [ ]:
# Extract metadata
meta = data['metadata']

# Build summary table
summary_data = {
    'Metric': [
        'Total Examples',
        'Typed Accuracy',
        'Baseline Accuracy',
        'Typed Hallucination Rate',
        'Baseline Hallucination Rate',
        'Typed Repairs Attempted',
        'Typed Repairs Succeeded',
        'Baseline Repairs Attempted',
        'Baseline Repairs Succeeded',
        'Total LLM Cost (USD)',
        'Total LLM Calls',
    ],
    'Value': [
        meta['total_examples'],
        f"{meta['typed_accuracy']:.1%}",
        f"{meta['baseline_accuracy']:.1%}",
        f"{meta['typed_hallucination_rate']:.1%}",
        f"{meta['baseline_hallucination_rate']:.1%}",
        meta['typed_repairs_attempted'],
        meta['typed_repairs_succeeded'],
        meta['baseline_repairs_attempted'],
        meta['baseline_repairs_succeeded'],
        f"${meta['total_cost_usd']:.4f}",
        meta['total_llm_calls'],
    ]
}

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*70)
print("TYPED FAILURE RECOVERY — FULL EXPERIMENT RESULTS")
print("="*70)
print(summary_df.to_string(index=False))
print("="*70)
print(f"\nKey Finding: {meta['key_finding']}")
print("="*70)


## 6. Visualization

Compare typed vs baseline performance across metrics.


In [ ]:
if VISUALIZE:
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle('Typed Failure Recovery vs Baseline', fontsize=14, fontweight='bold')
    
    # Accuracy comparison
    ax = axes[0, 0]
    methods = ['Typed', 'Baseline']
    accuracies = [meta['typed_accuracy'], meta['baseline_accuracy']]
    ax.bar(methods, accuracies, color=['#2E86AB', '#A23B72'])
    ax.set_ylabel('Accuracy')
    ax.set_title('Accuracy Comparison')
    ax.set_ylim(0, 1)
    for i, v in enumerate(accuracies):
        ax.text(i, v + 0.03, f'{v:.1%}', ha='center', fontweight='bold')
    
    # Hallucination comparison
    ax = axes[0, 1]
    hallucinations = [meta['typed_hallucination_rate'], meta['baseline_hallucination_rate']]
    ax.bar(methods, hallucinations, color=['#2E86AB', '#A23B72'])
    ax.set_ylabel('Hallucination Rate')
    ax.set_title('Hallucination Rate (lower is better)')
    ax.set_ylim(0, 0.15)
    for i, v in enumerate(hallucinations):
        ax.text(i, v + 0.005, f'{v:.1%}', ha='center', fontweight='bold')
    
    # Repair success rate
    ax = axes[1, 0]
    typed_repair_rate = meta['typed_repairs_succeeded'] / max(meta['typed_repairs_attempted'], 1)
    baseline_repair_rate = meta['baseline_repairs_succeeded'] / max(meta['baseline_repairs_attempted'], 1)
    repair_rates = [typed_repair_rate, baseline_repair_rate]
    ax.bar(methods, repair_rates, color=['#2E86AB', '#A23B72'])
    ax.set_ylabel('Repair Success Rate')
    ax.set_title(f'Repairs Succeeded / Attempted')
    ax.set_ylim(0, 1)
    for i, v in enumerate(repair_rates):
        ax.text(i, v + 0.03, f'{v:.1%}', ha='center', fontweight='bold')
    
    # Cost
    ax = axes[1, 1]
    ax.text(0.5, 0.5, f'Total Cost: \${meta["total_cost_usd"]:.4f}\n\nLLM Calls: {meta["total_llm_calls"]}\n\nModel: {meta["model"]}',
            ha='center', va='center', fontsize=12, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    ax.axis('off')
    
    plt.tight_layout()
    plt.show()
    logger.info("Visualization complete")


## 7. Sample Traces

First 5 examples from the full run showing per-example results.


In [ ]:
if VERBOSE:
    traces_df = pd.DataFrame(data['metadata']['sample_traces'])
    traces_df = traces_df[['example_id', 'dataset', 'expected', 'typed_result', 'baseline_result', 'failure_type']]
    print("\nSample Traces (first 5 examples):")
    print("="*100)
    print(traces_df.to_string(index=False))
    print("="*100)


## Conclusion

### Key Findings

1. **Typed repair accuracy (24%) vs baseline (26%)**: Both methods achieve similar overall accuracy, indicating that the primary failure mode is not repair strategy but **natural language-to-FOL extraction quality**.

2. **Zero hallucination (typed) vs 8% (baseline)**: The typed approach's conservative repair strategy avoids inventing facts not grounded in the source. The baseline trades accuracy for robustness, sometimes hallucinating bridging facts.

3. **Dominant failure type**: Type 1 (unknown predicate) accounts for most failures. The ProofWriter FOL parser generates predicate names that don't match the engine's extracted vocabulary — a **vocabulary alignment problem**, not a reasoning problem.

4. **Cost efficiency**: 86 LLM calls for 50 examples = $0.0296, well within the $8 budget. Haiku 4.5 provides sufficient reasoning capability at minimal cost.

### Implications

- **Neuro-symbolic pipeline viability**: The approach successfully integrates LLM repair with symbolic reasoning, demonstrating a path to verifiable, interpretable fact extraction.
- **FOL extraction is the bottleneck**: Investment in better NL→FOL translation (e.g., fine-tuning on ProofWriter/CLUTRR vocabulary) would likely yield significant gains.
- **Zero hallucination advantage**: In high-stakes domains (law, medicine), the typed approach's conservative strategy is valuable even if accuracy is comparable.

### Next Steps

1. Improve FOL vocabulary extraction with a learned predicate mapper
2. Apply the pipeline to real documents (legal contracts, news articles)
3. Extend kinship inference rules to multi-hop reasoning
4. Benchmark against pure RAG and chain-of-thought baselines
